# Pixels to Predictions — SmolVLM-500M + QLoRA Training

**Goal:** fine-tune `HuggingFaceTB/SmolVLM-500M-Instruct` on the ScienceQA-style multiple-choice dataset, targeting **80%+ test accuracy**.

**Constraints (from competition brief):**
- Base model fixed: `SmolVLM-500M-Instruct`
- ≤ 5M trainable parameters
- Only provided data, no external data
- Offline evaluation (no internet at inference)



## 1. Install dependencies

Run once per fresh runtime.

In [1]:
# Quiet install. Versions match the competition starter.
!pip install -q transformers==4.57.6 peft==0.18.1 accelerate==1.0.1 datasets pillow num2words

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 36.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.0/557.0 kB 73.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 330.9/330.9 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.5/163.5 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 78.7 MB/s eta 0:00:00


In [2]:
!pip install -U --no-cache-dir bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 434.5 MB/s eta 0:00:00


In [3]:
from google.colab import files
files.upload()   # kaggle api upload to download dataset

!mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!pip install -q kaggle

!mkdir -p /content/data
!cd /content/data && kaggle competitions download -c pixels-to-predictions --force
!cd /content/data && unzip -oq "*.zip" && rm -f *.zip
!ls /content/data

Saving kaggle.json to kaggle.json
100% 358M/358M [00:09<00:00, 38.9MB/s]

images	sample_submission.csv  test.csv  train.csv  val.csv


## 2. Imports, seed, paths

In [4]:
import os, json, random, math, gc, time
from pathlib import Path
from copy import deepcopy

import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoProcessor,
    AutoModelForVision2Seq,
    BitsAndBytesConfig,
    get_cosine_schedule_with_warmup,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)


DATA_DIR = Path("/content/data")
OUT_DIR  = Path("/content/outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_ID = "HuggingFaceTB/SmolVLM-500M-Instruct"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

device: cuda
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
VRAM: 102.0 GB


## 3. Load data

Loads `train.csv` and `test.csv`. If `val.csv` is present we use it; otherwise we hold out 10% of train for validation.

In [5]:
import os
# What's at the root content folder
!ls -la /content/
# Look for any csv files anywhere obvious
!find /content -maxdepth 3 -name "*.csv" 2>/dev/null
# Is Drive mounted?
!ls /content/drive/MyDrive/ 2>/dev/null | head

total 24
drwxr-xr-x 1 root root 4096 May  9 02:11 .
drwxr-xr-x 1 root root 4096 May  9 02:08 ..
drwxr-xr-x 4 root root 4096 May  6 13:32 .config
drwxr-xr-x 3 root root 4096 May  9 02:11 data
drwxr-xr-x 2 root root 4096 May  9 02:11 outputs
drwxr-xr-x 1 root root 4096 May  6 13:32 sample_data
/content/data/test.csv
/content/data/train.csv
/content/data/sample_submission.csv
/content/data/val.csv
/content/sample_data/california_housing_test.csv
/content/sample_data/mnist_test.csv
/content/sample_data/mnist_train_small.csv
/content/sample_data/california_housing_train.csv


In [6]:
train_df = pd.read_csv(DATA_DIR / "train.csv")
test_df  = pd.read_csv(DATA_DIR / "test.csv")

val_path = DATA_DIR / "val.csv"
if val_path.exists():
    val_df = pd.read_csv(val_path)
    print("Loaded provided val.csv")
else:
    # Stratified-ish split by num_choices to keep distribution similar
    rng = np.random.RandomState(SEED)
    idx = np.arange(len(train_df))
    rng.shuffle(idx)
    n_val = int(0.10 * len(train_df))
    val_df = train_df.iloc[idx[:n_val]].reset_index(drop=True)
    train_df = train_df.iloc[idx[n_val:]].reset_index(drop=True)
    print(f"No val.csv found — split train into {len(train_df)} train / {len(val_df)} val")

# Parse choices JSON
for df in [train_df, val_df, test_df]:
    df["choices"] = df["choices"].apply(json.loads)

train_df = pd.concat([train_df, val_df], ignore_index=True)
print(f"Combined train+val: {len(train_df)}")

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")
print("Train answer distribution:", train_df['answer'].value_counts().sort_index().to_dict())

Loaded provided val.csv
Combined train+val: 4157
Train: 4157 | Val: 1048 | Test: 1008
Train answer distribution: {0: 1471, 1: 1400, 2: 984, 3: 279, 4: 23}


## 4. Prompt template

Format follows the starter notebook (so the `<image>` token is in the right place for SmolVLM's processor) but with these tweaks:

- **Lecture / hint truncated to 800 chars** to keep sequences from blowing up.
- **Choices listed as `A. … / B. … / …`**.
- **Training target is just the answer letter** (e.g. ` B`) — short, loss-masking is clean.


In [7]:
CHOICE_LETTERS = "ABCDEFGHIJ"
MAX_LECTURE_CHARS = 800
MAX_HINT_CHARS    = 600

def _truncate(s, n):
    if s is None: return ""
    try:
        if isinstance(s, float) and pd.isna(s): return ""
    except: pass
    s = str(s)
    if s.lower().strip() == "nan": return ""
    return s[:n].rstrip()

def _meta_line(row):
    parts = []
    for k in ("subject", "topic", "category", "skill"):
        v = row.get(k, "")
        if v is not None and str(v).strip() and str(v).strip().lower() != "nan":
            parts.append(str(v).strip())
    return " | ".join(parts)

def build_prompt(row, choices=None, answer_idx=None):
    if choices is None:
        choices = row["choices"]
    ctx = []
    meta = _meta_line(row)
    if meta: ctx.append(f"Topic: {meta}")
    lec = _truncate(row.get("lecture",""), MAX_LECTURE_CHARS)
    hnt = _truncate(row.get("hint",""), MAX_HINT_CHARS)
    if lec.strip(): ctx.append(lec.strip())
    if hnt.strip(): ctx.append(hnt.strip())
    ctx_str = "\n".join(ctx)
    choices_str = "\n".join(f"  {CHOICE_LETTERS[i]}. {c}" for i, c in enumerate(choices))
    p = "<image>\n"
    if ctx_str: p += f"Context:\n{ctx_str}\n\n"
    p += f"Question: {row['question']}\n"
    p += f"Choices:\n{choices_str}\nAnswer:"
    if answer_idx is not None:
        p += f" {CHOICE_LETTERS[answer_idx]}"
    return p

print(build_prompt(train_df.iloc[0].to_dict(), answer_idx=int(train_df.iloc[0]['answer']))[:600])

<image>
Context:
Topic: natural science | literacy-in-science | Adaptations and natural selection | How can animal behaviors affect reproductive success? Identify evidence to support a claim
Animals increase their reproductive success when they have offspring that survive to reproduce.
Animals can increase their chances of having offspring by behaving in ways that help them get partners to mate and reproduce with. These partners are called mates. For example, animals may make special sounds, perform specific dances, or show off bright colors to attract mates. Animals may also compete with each


## 5. Dataset (with choice-order shuffling)

During training we shuffle the order of choices and remap the answer letter. This kills the position bias we saw in the data (answer=0 appears 36% of the time).

In [8]:
class ScienceQADataset(Dataset):
    def __init__(self, df, data_dir, is_train=True, shuffle_choices=True):
        self.df = df.reset_index(drop=True)
        self.data_dir = Path(data_dir)
        self.is_train = is_train
        self.shuffle_choices = shuffle_choices and is_train

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(self.data_dir / row["image_path"]).convert("RGB")
        if self.is_train:
          from torchvision.transforms import functional as TF
          img = TF.adjust_brightness(img, 0.9 + 0.2 * random.random())
          img = TF.adjust_contrast(img,  0.9 + 0.2 * random.random())
          if random.random() < 0.5:
              img = TF.rotate(img, random.uniform(-3, 3), fill=255)

        choices = list(row["choices"])
        answer  = int(row["answer"]) if "answer" in row and pd.notna(row.get("answer", None)) else -1

        # Shuffle choices on training samples
        if self.shuffle_choices and answer >= 0:
            order = list(range(len(choices)))
            random.shuffle(order)
            choices = [choices[i] for i in order]
            answer  = order.index(int(row["answer"]))  # new position of the gold answer

        item = {
    "id": row["id"],
    "image": img,
    "row": row.to_dict(),
    "choices": choices,
    "num_choices": len(choices),
    "answer": answer,
              }
        return item

train_ds = ScienceQADataset(train_df, DATA_DIR, is_train=True,  shuffle_choices=True)
val_ds   = ScienceQADataset(val_df,   DATA_DIR, is_train=False, shuffle_choices=False)
test_ds  = ScienceQADataset(test_df,  DATA_DIR, is_train=False, shuffle_choices=False)
print(f"datasets ok — train={len(train_ds)} val={len(val_ds)} test={len(test_ds)}")

datasets ok — train=4157 val=1048 test=1008


## 6. Load SmolVLM in 4-bit + apply LoRA

LoRA is applied **only** to the text decoder's linear layers (not vision tower, not connector). With rank 8 across the 7 linear types in 32 layers we land at ≈ 4.34M trainable params — under the 5M cap.

In [9]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

processor = AutoProcessor.from_pretrained(MODEL_ID)
# SmolVLM default is fine; explicitly disable image splitting → 1 image, faster
processor.image_processor.do_image_splitting = False
# Resize so the longest edge is 384 (SmolVLM's native is 512, but 384 is plenty)
processor.image_processor.size = {"longest_edge": 384}
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token
# Right padding for causal LM training; we'll switch to left-pad for generation later
processor.tokenizer.padding_side = "right"

model = AutoModelForVision2Seq.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16,
    device_map={"": 0},
    low_cpu_mem_usage=True,
)
model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
)

# Find target modules: only the text decoder's q/k/v/o/gate/up/down projections
TEXT_LINEAR_SUFFIXES = {"q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"}
target_modules = []
for name, mod in model.named_modules():
    if isinstance(mod, torch.nn.Linear) and "text_model" in name:
        if name.split(".")[-1] in TEXT_LINEAR_SUFFIXES:
            target_modules.append(name)
print(f"# target linear modules: {len(target_modules)}  (e.g. {target_modules[0]})")

lora_cfg = LoraConfig(
    r=9,
    lora_alpha=18,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=target_modules,   # explicit list → no risk of hitting vision tower
)
model = get_peft_model(model, lora_cfg)

# Belt-and-braces: make sure vision encoder + connector are frozen
for n, p in model.named_parameters():
    if "vision_model" in n or "connector" in n:
        p.requires_grad = False

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"trainable: {trainable/1e6:.2f}M  /  total: {total/1e6:.2f}M  ({100*trainable/total:.3f}%)")
assert trainable < 5_000_000, "Trainable param count exceeds 5M cap!"
print("✓ under 5M cap")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


processor_config.json:   0%|          | 0.00/68.0 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/429 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/486 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

config.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/1.02G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/136 [00:00<?, ?B/s]

# target linear modules: 224  (e.g. model.text_model.layers.0.self_attn.q_proj)
trainable: 4.88M  /  total: 306.71M  (1.593%)
✓ under 5M cap


## 7. Letter token IDs

We need the token id for each `' A'`, `' B'`, … so we can (a) properly mask loss to just the answer token and (b) score by single-forward-pass at inference.

In [10]:
def get_letter_token_id(letter):
    """Return the single token id for ' <LETTER>' (with leading space)."""
    ids = processor.tokenizer.encode(" " + letter, add_special_tokens=False)
    assert len(ids) == 1, f"' {letter}' did not tokenize to a single token: {ids}"
    return ids[0]

LETTER_TOKEN_IDS = [get_letter_token_id(L) for L in CHOICE_LETTERS[:5]]  # A..E (max 5 choices)
print("Letter token ids (A..E):", LETTER_TOKEN_IDS)
print("Decoded check:", [processor.tokenizer.decode([t]) for t in LETTER_TOKEN_IDS])

Letter token ids (A..E): [330, 389, 340, 422, 414]
Decoded check: [' A', ' B', ' C', ' D', ' E']


## 8. Collator

Builds a single batch from items returned by the dataset:

1. Builds the **full** training text (prompt + ` <LETTER>`) for each example.
2. Sends it through the processor with the image.
3. Constructs `labels`: `-100` everywhere except the **answer letter token** at the very end of each (non-padded) sequence.


In [11]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path("/content/data")
print("=== top-level contents of DATA_DIR ===")
!ls -la {DATA_DIR}
print("\n=== anything containing 'image' under DATA_DIR (depth 3) ===")
!find {DATA_DIR} -maxdepth 3 -type d -iname "*image*"
print("\n=== count of png files anywhere under DATA_DIR ===")
!find {DATA_DIR} -name "*.png" 2>/dev/null | wc -l
print("\n=== where (if anywhere) is train_01835.png? ===")
!find {DATA_DIR} -name "train_01835.png" 2>/dev/null
print("\n=== first three image_path values from train.csv ===")
print(pd.read_csv(DATA_DIR / "train.csv")["image_path"].head(3).tolist())
print("\n=== are there any zip files left to unpack? ===")
!find {DATA_DIR} -maxdepth 2 -name "*.zip"

=== top-level contents of DATA_DIR ===
total 7528
drwxr-xr-x 3 root root    4096 May  9 02:11 .
drwxr-xr-x 1 root root    4096 May  9 02:11 ..
drwxr-xr-x 3 root root    4096 May  9 02:11 images
-rw-r--r-- 1 root root   13114 Apr 22 17:30 sample_submission.csv
-rw-r--r-- 1 root root 1214269 Apr 22 17:30 test.csv
-rw-r--r-- 1 root root 4844569 Apr 22 17:30 train.csv
-rw-r--r-- 1 root root 1616229 Apr 22 17:30 val.csv

=== anything containing 'image' under DATA_DIR (depth 3) ===
/content/data/images
/content/data/images/images

=== count of png files anywhere under DATA_DIR ===
5165

=== where (if anywhere) is train_01835.png? ===
/content/data/images/images/train/train_01835.png

=== first three image_path values from train.csv ===
['images/train/train_07667.png', 'images/train/train_02628.png', 'images/train/train_00927.png']

=== are there any zip files left to unpack? ===


In [12]:
!mv /content/data/images/images/* /content/data/images/
!rmdir /content/data/images/images
!ls /content/data/images           # should show: train  test  val
!ls /content/data/images/train/train_07667.png   # should print the path, no error

test  train  val
/content/data/images/train/train_07667.png


In [13]:
def train_collator(batch):
    texts, images, ans_letters = [], [], []
    for item in batch:
        texts.append(build_prompt(item["row"], choices=item["choices"], answer_idx=item["answer"]))
        images.append(item["image"])
        ans_letters.append(CHOICE_LETTERS[item["answer"]])

    inputs = processor(text=texts, images=images, padding=True, return_tensors="pt")

    input_ids = inputs["input_ids"]
    attn      = inputs["attention_mask"]
    labels    = torch.full_like(input_ids, -100)
    for i in range(input_ids.size(0)):
        last = attn[i].sum().item() - 1
        assert input_ids[i, last].item() in LETTER_TOKEN_IDS, \
            f"Last token isn't a letter id: {input_ids[i, last].item()}"
        labels[i, last] = input_ids[i, last]
    inputs["labels"] = labels
    return inputs

# Smoke test
sample_batch = [train_ds[i] for i in range(2)]
b = train_collator(sample_batch)
print({k: (v.shape if torch.is_tensor(v) else type(v).__name__) for k,v in b.items()})
print("non-(-100) labels per row:", (b['labels']!=-100).sum(dim=1).tolist())

{'pixel_values': torch.Size([2, 1, 3, 512, 512]), 'pixel_attention_mask': torch.Size([2, 1, 512, 512]), 'input_ids': torch.Size([2, 477]), 'attention_mask': torch.Size([2, 477]), 'labels': torch.Size([2, 477])}
non-(-100) labels per row: [1, 1]


## 9. Train

Custom loop (cleaner than `Trainer` for this VLM setup). Per epoch:
1. Train one full pass with grad accumulation.
2. Run validation accuracy via single-forward-pass MCQ scoring.
3. Save the best LoRA adapter.


In [14]:
PER_DEVICE_BATCH = 4
GRAD_ACCUM       = 4
EFF_BATCH        = PER_DEVICE_BATCH * GRAD_ACCUM
LR               = 2e-4
NUM_EPOCHS       = 5
WARMUP_RATIO     = 0.05
WEIGHT_DECAY     = 0.0
MAX_GRAD_NORM    = 1.0

steps_per_epoch  = math.ceil(len(train_ds) / EFF_BATCH)
total_steps      = steps_per_epoch * NUM_EPOCHS
warmup_steps     = int(WARMUP_RATIO * total_steps)
print(f"steps/epoch={steps_per_epoch}  total={total_steps}  warmup={warmup_steps}")

import bitsandbytes as bnb
optim = bnb.optim.PagedAdamW8bit(
    [p for p in model.parameters() if p.requires_grad],
    lr=LR, betas=(0.9, 0.999), eps=1e-8, weight_decay=WEIGHT_DECAY,
)
scheduler = get_cosine_schedule_with_warmup(optim, warmup_steps, total_steps)

train_loader = DataLoader(
    train_ds, batch_size=PER_DEVICE_BATCH, shuffle=True,
    num_workers=2, collate_fn=train_collator, pin_memory=True, drop_last=False,
)

steps/epoch=260  total=1300  warmup=65


### Validation eval (single-forward-pass MCQ)

Score each test/val example by a **single forward pass**: tokenize the prompt up to `Answer:`, take the next-token logits, restrict to just the candidate letter token ids, argmax.

In [15]:
@torch.no_grad()
def predict_letter_logits(items, max_batch=8):
    out = []
    model.eval()
    old_side = processor.tokenizer.padding_side
    processor.tokenizer.padding_side = "left"
    try:
        for s in range(0, len(items), max_batch):
            chunk = items[s:s+max_batch]
            texts = [build_prompt(it["row"], choices=it["choices"], answer_idx=None) for it in chunk]
            images = [it["image"] for it in chunk]
            inputs = processor(text=texts, images=images, padding=True, return_tensors="pt").to(device)
            with torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16):
                logits = model(**inputs).logits
            last_logits = logits[:, -1, :]
            for i, it in enumerate(chunk):
                k = it["num_choices"]
                ids = LETTER_TOKEN_IDS[:k]
                out.append((last_logits[i, ids].float().cpu().numpy(), k))
    finally:
        processor.tokenizer.padding_side = old_side
        model.train()
    return out

def evaluate(ds, max_batch=8):
    items = [ds[i] for i in range(len(ds))]
    scored = predict_letter_logits(items, max_batch=max_batch)
    correct = 0
    for it, (lg, k) in zip(items, scored):
        pred = int(np.argmax(lg))
        if pred == it["answer"]:
            correct += 1
    return correct / len(items)

### Training loop

In [16]:
best_val = -1.0
best_dir = OUT_DIR / "lora_best"
log_every = 25
global_step = 0

model.train()
for epoch in range(1, NUM_EPOCHS + 1):
    epoch_t0 = time.time()
    optim.zero_grad(set_to_none=True)
    running_loss, running_n = 0.0, 0

    for step, batch in enumerate(train_loader):
        batch = {k: (v.to(device, non_blocking=True) if torch.is_tensor(v) else v)
                 for k, v in batch.items()}
        with torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16):
            out = model(**batch)
            loss = out.loss / GRAD_ACCUM
        loss.backward()
        running_loss += out.loss.item()
        running_n    += 1

        if (step + 1) % GRAD_ACCUM == 0:
            torch.nn.utils.clip_grad_norm_(
                [p for p in model.parameters() if p.requires_grad], MAX_GRAD_NORM)
            optim.step()
            scheduler.step()
            optim.zero_grad(set_to_none=True)
            global_step += 1
            if global_step % log_every == 0:
                lr = scheduler.get_last_lr()[0]
                print(f"  ep{epoch} step{global_step}/{total_steps}  loss={running_loss/running_n:.4f}  lr={lr:.2e}")

    # End-of-epoch eval
    val_acc = evaluate(val_ds, max_batch=8)
    dt = time.time() - epoch_t0
    print(f"[epoch {epoch}] val_acc = {val_acc:.4f}   time = {dt/60:.1f} min")

    if val_acc > best_val:
        best_val = val_acc
        model.save_pretrained(best_dir)          # saves only the LoRA adapter
        processor.save_pretrained(best_dir)
        print(f"  ↑ new best ({best_val:.4f}) saved to {best_dir}")

print(f"\nBEST val acc: {best_val:.4f}")

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...


  ep1 step25/1300  loss=1.3974  lr=7.69e-05
  ep1 step50/1300  loss=1.1735  lr=1.54e-04
  ep1 step75/1300  loss=1.0622  lr=2.00e-04
  ep1 step100/1300  loss=0.9802  lr=2.00e-04
  ep1 step125/1300  loss=0.9097  lr=1.99e-04
  ep1 step150/1300  loss=0.8730  lr=1.98e-04
  ep1 step175/1300  loss=0.8325  lr=1.96e-04
  ep1 step200/1300  loss=0.8026  lr=1.94e-04
  ep1 step225/1300  loss=0.7884  lr=1.92e-04
  ep1 step250/1300  loss=0.7707  lr=1.89e-04
[epoch 1] val_acc = 0.7300   time = 4.6 min
  ↑ new best (0.7300) saved to /content/outputs/lora_best
  ep2 step275/1300  loss=0.4542  lr=1.86e-04
  ep2 step300/1300  loss=0.4760  lr=1.83e-04
  ep2 step325/1300  loss=0.4819  lr=1.79e-04
  ep2 step350/1300  loss=0.4772  lr=1.75e-04
  ep2 step375/1300  loss=0.4810  lr=1.70e-04
  ep2 step400/1300  loss=0.4897  lr=1.66e-04
  ep2 step425/1300  loss=0.4904  lr=1.61e-04
  ep2 step450/1300  loss=0.4800  lr=1.56e-04
  ep2 step475/1300  loss=0.4782  lr=1.50e-04
  ep2 step500/1300  loss=0.4791  lr=1.45e-04
[

## 10. Inference on test → `submission.csv`

Reload the **best** LoRA adapter, then score every test example by single-forward-pass MCQ. Optionally enable choice-permutation TTA — gives an extra ~0.5–1.5 points for free at the cost of K× the compute.

In [17]:
# Reload best adapter cleanly
del model; gc.collect(); torch.cuda.empty_cache()

base = AutoModelForVision2Seq.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16,
    device_map={"": 0},
    low_cpu_mem_usage=True,
)
model = PeftModel.from_pretrained(base, str(best_dir))
model.eval()

USE_TTA = True
TTA_PERMUTATIONS = 8      # number of choice orderings to average over

@torch.no_grad()
def predict_test():
    preds, ids = [], []
    items = [test_ds[i] for i in range(len(test_ds))]

    if not USE_TTA:
        scored = predict_letter_logits(items, max_batch=8)
        for it, (lg, k) in zip(items, scored):
            ids.append(it["id"])
            preds.append(int(np.argmax(lg)))
        return ids, preds

    # TTA: average softmax probs across several random permutations of choices
    rng = random.Random(SEED)
    n = len(items)
    K_max = max(it["num_choices"] for it in items)
    accum = [np.zeros(it["num_choices"], dtype=np.float64) for it in items]

    for t in range(TTA_PERMUTATIONS):
        permuted_items = []
        perms = []
        for it in items:
            k = it["num_choices"]
            perm = list(range(k))
            if t > 0:                       # 1st pass = identity ordering
                rng.shuffle(perm)
            perms.append(perm)
            new_it = dict(it)
            new_it["choices"] = [it["choices"][j] for j in perm]
            permuted_items.append(new_it)

        scored = predict_letter_logits(permuted_items, max_batch=8)
        for i, ((lg, k), perm) in enumerate(zip(scored, perms)):
            probs = np.exp(lg - lg.max())
            probs /= probs.sum()
            # Un-permute back to original choice order
            unperm = np.zeros_like(probs)
            for new_pos, orig_pos in enumerate(perm):
                unperm[orig_pos] = probs[new_pos]
            accum[i] += unperm
        print(f"  TTA pass {t+1}/{TTA_PERMUTATIONS} done")

    for it, p in zip(items, accum):
        ids.append(it["id"])
        preds.append(int(np.argmax(p)))
    return ids, preds

ids, preds = predict_test()
sub = pd.DataFrame({"id": ids, "answer": preds})


sample = pd.read_csv(DATA_DIR / "sample_submission.csv") if (DATA_DIR/"sample_submission.csv").exists() else None
if sample is not None:
    sub = sample[["id"]].merge(sub, on="id", how="left")
    assert sub["answer"].notna().all(), "Some test ids are missing predictions!"
    sub["answer"] = sub["answer"].astype(int)

sub_path = OUT_DIR / "submission.csv"
sub.to_csv(sub_path, index=False)
print(f"Wrote {sub_path}  ({len(sub)} rows)")
print(sub.head())
print("\nPred distribution:", sub['answer'].value_counts().sort_index().to_dict())


/usr/local/lib/python3.12/dist-packages/transformers/models/auto/modeling_auto.py:2284: FutureWarning: The class `AutoModelForVision2Seq` is deprecated and will be removed in v5.0. Please use `AutoModelForImageTextToText` instead.
  warnings.warn(


  TTA pass 1/8 done
  TTA pass 2/8 done
  TTA pass 3/8 done
  TTA pass 4/8 done
  TTA pass 5/8 done
  TTA pass 6/8 done
  TTA pass 7/8 done
  TTA pass 8/8 done
Wrote /content/outputs/submission.csv  (1008 rows)
           id  answer
0  test_01750       2
1  test_00128       0
2  test_02891       0
3  test_02425       1
4  test_00930       2

Pred distribution: {0: 348, 1: 364, 2: 216, 3: 75, 4: 5}
